## 02 — The Comparison

This is the final notebook.

We put both approaches side by side: the system we built across seven modules and the one command that replaces most of it. The goal is not to show that `tippecanoe` is better — it obviously is for production use — but to show exactly **what work it is doing**, because we built every one of those pieces ourselves.

## System Comparison — Architecture

```
OUR SYSTEM                                  TIPPECANOE + TILE CLIENT
─────────────────────────────────────       ──────────────────────────────────────
ne_10m_railroads.geojson (40 MB)            ne_10m_railroads.geojson (40 MB)
       │                                           │
       ▼                                           ▼
Module 02: simplify at 4 epsilons           tippecanoe (one command, ~30 seconds)
  → 4 GeoJSON output files                        │
       │                                           ▼
       ▼                                    railroads.pmtiles (~3 MB)
Module 04: build 4 GridIndex objects               │
  (startup: ~5 seconds)                            ▼
       │                               tile client (browser or localtileserver)
       ▼                                 fetches only visible tiles on demand
Module 05: get_lod(zoom) selects index
       │
       ▼
Module 03: bbox cull within selected index
       │
       ▼
GeoJSON layer (re-sent every pan/zoom)
```

## System Comparison — Numbers

In [3]:
from pathlib import Path


def find_data_dir():
    """Find the Data Manager data directory from common notebook launch locations."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates = [
            base / "data",
            base / "03-Data_Manager" / "data",
            base / "Assignments_Completed" / "03-Data_Manager" / "data",
        ]
        for candidate in candidates:
            if (candidate / "ne_10m_railroads.geojson").exists() and (candidate / "lod").exists():
                return candidate
    raise FileNotFoundError("Could not find Assignments_Completed/03-Data_Manager/data")


data_dir = find_data_dir()
lod_dir = data_dir / "lod"
raw     = data_dir / "ne_10m_railroads.geojson"
pmtiles = data_dir / "railroads.pmtiles"

lod_files = [
    "railroads_coarse.geojson",
    "railroads_medium.geojson",
    "railroads_fine.geojson",
    "railroads_extra_fine.geojson",
]

our_total_mb = sum((lod_dir / f).stat().st_size for f in lod_files) / 1_000_000
raw_mb       = raw.stat().st_size / 1_000_000
pm_mb        = pmtiles.stat().st_size / 1_000_000 if pmtiles.exists() else None

print("Storage comparison:")
print(f"  Raw GeoJSON:               {raw_mb:.1f} MB")
print(f"  Our 4 LOD files (total):   {our_total_mb:.1f} MB")
if pm_mb:
    print(f"  tippecanoe PMTiles:        {pm_mb:.1f} MB")
else:
    print("  tippecanoe PMTiles:        (run Notebook 01 first)")

print()
print("Runtime comparison:")
print(f"  Our startup (load + index): ~5–10s")
print(f"  Our per-query time:         ~1–5ms")
print(f"  Tile client startup:        ~0s (lazy fetch)")
print(f"  Tile fetch (per tile):      ~10–50ms over network")
print(f"  Tile fetch (local):         <1ms")


Storage comparison:
  Raw GeoJSON:               39.6 MB
  Our 4 LOD files (total):   41.1 MB
  tippecanoe PMTiles:        (run Notebook 01 first)

Runtime comparison:
  Our startup (load + index): ~5–10s
  Our per-query time:         ~1–5ms
  Tile client startup:        ~0s (lazy fetch)
  Tile fetch (per tile):      ~10–50ms over network
  Tile fetch (local):         <1ms


## What Tippecanoe Automated — Specifically

Now name each thing precisely — because you built it:

**1. Multi-resolution simplification (our Module 02)**
tippecanoe applies a tolerance appropriate for each zoom level automatically. You do not choose 4 epsilons — you choose one `--simplification` value and it scales it per zoom.

**2. Spatial bucketing (our Module 04)**
Tiles are the index. There is no separate grid index to build — the tile `(z, x, y)` address is the bucket. Features are pre-assigned to tiles at generation time.

**3. Viewport culling (our Module 03)**
The client requests only the tiles its viewport covers. No intersection test — irrelevant tiles are never fetched.

**4. LOD switching (our Module 05)**
The tile URL includes the zoom level (`/{z}/`). The client naturally requests tiles at the right zoom. No decision function needed.

**5. Binary encoding**
We never built this — our GeoJSON is text. Tippecanoe outputs MVT binary with integer coordinates. ~5× smaller and faster to parse.

**6. Streaming / lazy loading**
We never built this either. Tiles are fetched on demand; unused regions are never touched.

## What Tippecanoe Does NOT Do

Tippecanoe is a **preprocessing pipeline** — it generates static files. It does not:

- Serve tiles dynamically (you still need a static host, CDN, or `localtileserver`)
- Filter features at query time based on user input
- Handle real-time data updates
- Decide which features to show based on screen density or user preferences at render time

For live, user-driven filtering (e.g., "show only electrified railways"), a tile system either:
- Pre-generates multiple tile sets (one per filter combination)
- Sends all attributes in the tile and lets the client filter at render time (style expressions)
- Uses a dynamic tile server that queries a database per request

None of these are trivial. Our handbuilt system could add real-time filtering in ten lines.

## The Quote That Started This

> *"Build the smallest version that teaches the idea. Borrow the version that survives the real world."*

You built the smallest version. You know:
- What Douglas-Peucker does and why epsilon matters
- Why spatial indexes exist and what they trade off
- Why bounding box culling is O(n) without an index
- Why the tile coordinate scheme is itself a spatial index
- Why binary encoding is worth the complexity

When you use `tippecanoe` from now on, you can read its flags without guessing. When it produces unexpected output, you can reason about why. When a colleague says "we should just use vector tiles" without understanding the tradeoffs, you can ask the right questions.

That is not the same as having typed `tippecanoe` once.

## Exercise A

Write a one-page (or ~15 bullet point) technical comparison of the two systems. Cover:
- Build time
- Storage footprint
- Startup cost for the end user
- Per-request data transfer
- Support for real-time filtering
- Complexity to maintain
- What you would use for a class project vs. a production application

Write it in the cell below as markdown.

- Build time: our system takes several notebooks and several custom modules to generate simplified GeoJSON, build indexes, and wire the viewer together. Tippecanoe does the preprocessing in one command once the tool is installed.
- Storage footprint: the raw railroad GeoJSON is about 39.6 MB. Our four LOD GeoJSON files total about 41.1 MB, while the PMTiles output is expected to be much smaller because it stores vector tiles in compact binary form.
- Startup cost: our viewer has to load LOD files and build or load spatial indexes before it can answer map requests. A tile client starts almost immediately because it only asks for the visible tiles.
- Per-request data transfer: our system can still send GeoJSON chunks for each viewport, which are text-heavy. Tippecanoe sends small binary MVT tiles for just the requested `{z}/{x}/{y}` tile addresses.
- Simplification: our system uses four hand-picked epsilon values. Tippecanoe applies simplification per zoom level, so detail changes more smoothly across the tile pyramid.
- Spatial indexing: our system builds `GridIndex` objects to reduce search time. Tippecanoe bakes the spatial index into the tile pyramid itself, where each tile is already a bucket.
- Viewport culling: our system computes bounding-box intersections against the selected index. A tile client avoids irrelevant data by never requesting tiles outside the viewport.
- LOD switching: our system needs a decision function like `get_lod(zoom)`. Tippecanoe uses the zoom level in the tile URL, so the client naturally requests the matching level.
- Runtime filtering: our handbuilt system is easier to modify for live filters because the Python code can inspect feature properties at request time. Static Tippecanoe tiles cannot invent a new filtered dataset unless the needed attributes/features were encoded ahead of time.
- Attribute filtering: Tippecanoe can keep or drop attributes during tile generation, but that is a preprocessing decision. If an attribute is excluded from the tile, the browser cannot style or filter on it later.
- Maintenance complexity: our system is easier to study but has more custom code to maintain, test, and optimize. Tippecanoe shifts that complexity into a mature external tool.
- Debugging complexity: our system exposes every step, which helps explain bugs. Tippecanoe is more compact, but debugging requires understanding flags, tile metadata, zoom behavior, and dropped features.
- Production scalability: Tippecanoe is better for production because PMTiles/MVT is compact, cacheable, CDN-friendly, and works with existing map clients.
- Class project fit: I would use our handbuilt system for a class project because it teaches simplification, indexing, culling, and LOD decisions directly.
- Production fit: I would use Tippecanoe for a production application unless I needed live server-side filtering or constantly changing data that required dynamic tile generation.


## Exercise B

Look up the `--attribute-filter` and `--include` flags in the tippecanoe documentation.

Could you use these to produce a tile set that only includes electrified railroads (`electric` property)? Write the command you would use, and explain what the resulting tile set would and would not be able to show.

In [ ]:
# Build a PMTiles file containing only electrified railroads.
# In this Natural Earth file, electric == 1 appears to mark electrified lines.
# --feature-filter removes non-matching features; --include keeps only the attributes
# that the tile client still needs for styling/filtering after the tiles are built.

# tippecanoe \
#   --output=../../data/railroads_electric.pmtiles \
#   --force \
#   --minimum-zoom=1 \
#   --maximum-zoom=14 \
#   --simplification=10 \
#   --drop-densest-as-needed \
#   --layer=railroads \
#   --include=electric \
#   --include=scalerank \
#   --feature-filter='{ "*": [ "==", "electric", 1 ] }' \
#   ../../data/ne_10m_railroads.geojson

# Result: the tile set can show the geometry of electrified railroads and can
# still style/filter by the included electric and scalerank attributes.
# It cannot show non-electrified railroads, compare electrified vs. non-electrified
# lines, or filter/style by attributes that were not preserved with --include.
# The attribute-filter expression is different: it conditionally removes an
# attribute from matching features, but it does not remove the feature itself.


## Check Your Understanding

A classmate who skipped Modules 01–06 and came straight to this notebook could run `tippecanoe` and get a working tile set. They would see the flags but not know what they mean.

Name **three specific situations** where their lack of understanding would cost them — where they would make a wrong decision, miss a bug, or be unable to debug a problem — that you would be able to handle.

---

1. They might choose a simplification value that looks fine globally but destroys local geometry at high zoom. Because we built Douglas-Peucker examples, we can connect `--simplification` to the actual loss of vertices and shape detail.

2. They might think `--drop-densest-as-needed` is a semantic importance filter and assume important railroads are preserved. Because we built scalerank filtering and density-aware culling separately, we know density-based dropping is about tile size and visual crowding, not railroad importance.

3. They might exclude an attribute with `--include` or an attribute filter and later wonder why browser-side styling or filtering no longer works. Because we built the GeoJSON property workflow ourselves, we know that once an attribute is left out of the vector tile, the client cannot use it.

4. They might debug slow panning in the wrong place. Without the earlier modules, they may not understand whether the bottleneck is raw data size, missing spatial culling, too many visible features, tile size, or client rendering.

5. They might expect static PMTiles to support real-time user-driven queries. Because we built the handrolled system, we know the difference between preprocessing data into tiles and querying/filtering features dynamically at request time.


## End of the Data Manager Micro Lessons

You built:
- A simplification algorithm
- A multi-resolution data pipeline
- A spatial intersection test
- A grid-based spatial index
- A zoom-driven data selection system
- A working interactive map viewer
- An informed opinion about when to stop building and borrow instead

The railroad project is where you apply it.